# Lectura de los datos descargados

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'mitma').is_dir():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'mitma'
assert DATA.is_dir(), 'Abre el notebook desde project/ o project/notebooks/.'

con = duckdb.connect(':memory:')
print('Carpeta de datos:', DATA)

Carpeta de datos: /home/pablo/Desktop/muceim/Big Data Engineering/project/data/mitma


## 1. Archivos disponibles

Mostramos todos los archivos con su tamaño, sin leer todavía su contenido.

In [2]:
archivos = pd.DataFrame([
    {'archivo': str(p.relative_to(DATA)), 'MiB': round(p.stat().st_size / 2**20, 2)}
    for p in sorted(DATA.rglob('*')) if p.is_file()
])
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(archivos)

,archivo,MiB
0,calidad/diarios/20230101_distritos_descartados.csv,0.00
1,por-GAU/pernoctaciones/diarios/20230101_Pernoctaciones_GAU.csv.gz,2.43
2,por-GAU/personas/diarios/20230101_Personas_dia_GAU.csv.gz,0.42
3,por-GAU/viajes/diarios/20230101_Viajes_GAU.csv.gz,72.78
4,por-distritos/pernoctaciones/diarios/20230101_Pernoctaciones_distritos.csv.gz,3.70
5,por-distritos/personas/diarios/20230101_Personas_dia_distritos.csv.gz,0.78
6,por-distritos/viajes/diarios/20230101_Viajes_distritos.csv.gz,149.26
7,por-municipios/pernoctaciones/diarios/20230101_Pernoctaciones_municipios.csv.gz,2.89
8,por-municipios/personas/diarios/20230101_Personas_dia_municipios.csv.gz,0.53
9,por-municipios/viajes/diarios/20230101_Viajes_municipios.csv.gz,108.89


## 2. Elegir el nivel territorial y la fecha

Cambia estos valores para abrir otros archivos descargados. Los niveles disponibles son `municipios`, `distritos` y `GAU`.

In [3]:
NIVEL = 'municipios'
FECHA = '20230101'
FILAS = 5

assert NIVEL in {'municipios', 'distritos', 'GAU'}

def leer_primeras_filas(ruta, cabecera=True):
    # Los códigos quedan como texto y conservan sus ceros iniciales.
    # No interpretamos NA ni los campos vacíos como valores nulos.
    return con.execute(
        "SELECT * FROM read_csv(?, delim='|', header=?, all_varchar=true, "
        "nullstr='__NULL_NO_PUBLICADO__') LIMIT ?",
        [str(ruta), cabecera, FILAS],
    ).df()

## 3. Viajes

Cada fila contiene origen, destino, periodo, categorías publicadas y medidas de viajes. Aquí solo las leemos: todas las columnas se mantienen como texto.

In [4]:
ruta = DATA / f'por-{NIVEL}' / 'viajes' / 'diarios' / f'{FECHA}_Viajes_{NIVEL}.csv.gz'
viajes = leer_primeras_filas(ruta)
display(viajes)
print('Columnas:', ', '.join(viajes.columns))

,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
0,20230101,00,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.922,12.806
1,20230101,00,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NA,NA,5.202,22.098
2,20230101,00,01001,01009_AM,0.5-2,frecuente,casa,no,no,01,>15,NA,NA,1.0,1.0
3,20230101,00,01001,01009_AM,0.5-2,no_frecuente,no_frecuente,no,no,01,10-15,NA,NA,2.33639833318,2.65770202355
4,20230101,00,01001,01009_AM,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.35400937589,13.7553984279


Columnas: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km


In [16]:
cols1 = viajes.columns

In [17]:
ruta = DATA / f'por-distritos' / 'viajes' / 'diarios' / f'{FECHA}_Viajes_distritos.csv.gz'
viajes = leer_primeras_filas(ruta)
display(viajes)
print('Columnas:', ', '.join(viajes.columns))

,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
0,20230101,00,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.922,12.806
1,20230101,00,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NA,NA,5.202,22.098
2,20230101,01,01001,01001,2-10,no_frecuente,no_frecuente,no,no,31,>15,25-45,NA,4.648,20.805
3,20230101,02,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.28,9.213
4,20230101,03,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NA,NA,2.922,11.72


Columnas: fecha, periodo, origen, destino, distancia, actividad_origen, actividad_destino, estudio_origen_posible, estudio_destino_posible, residencia, renta, edad, sexo, viajes, viajes_km


In [18]:
cols2 = viajes.columns

In [19]:
print(cols1)
print(cols2)

Index(['fecha', 'periodo', 'origen', 'destino', 'distancia',
       'actividad_origen', 'actividad_destino', 'estudio_origen_posible',
       'estudio_destino_posible', 'residencia', 'renta', 'edad', 'sexo',
       'viajes', 'viajes_km'],
      dtype='str')
Index(['fecha', 'periodo', 'origen', 'destino', 'distancia',
       'actividad_origen', 'actividad_destino', 'estudio_origen_posible',
       'estudio_destino_posible', 'residencia', 'renta', 'edad', 'sexo',
       'viajes', 'viajes_km'],
      dtype='str')


## 4. Personas

Personas por zona de pernoctación, edad, sexo y categoría de número de viajes.

In [5]:
ruta = DATA / f'por-{NIVEL}' / 'personas' / 'diarios' / f'{FECHA}_Personas_dia_{NIVEL}.csv.gz'
personas = leer_primeras_filas(ruta)
display(personas)

,fecha,zona_pernoctacion,edad,sexo,numero_viajes,personas
0,20230101,01001,0-25,hombre,0,251.165
1,20230101,01001,0-25,hombre,1,53.172
2,20230101,01001,0-25,hombre,2,59.478
3,20230101,01001,0-25,hombre,2+,96.707
4,20230101,01001,0-25,mujer,0,201.435


## 5. Pernoctaciones

Personas por zona de residencia y zona de pernoctación.

In [6]:
ruta = DATA / f'por-{NIVEL}' / 'pernoctaciones' / 'diarios' / f'{FECHA}_Pernoctaciones_{NIVEL}.csv.gz'
pernoctaciones = leer_primeras_filas(ruta)
display(pernoctaciones)

,fecha,zona_residencia,zona_pernoctacion,personas
0,20230101,01001,01001,2329.644
1,20230101,01001,01017_AM,16.682
2,20230101,01001,01028_AM,2.922
3,20230101,01001,01051,6.056
4,20230101,01001,01054_AM,5.897


## 6. Nombres y población de las zonas

Los CSV de población no tienen cabecera: usamos `cabecera=False` para no perder la primera fila. Los nombres municipales incluyen una columna índice original que conservamos. El año de esta población de referencia no está verificado.

In [7]:
sufijo = 'gaus' if NIVEL == 'GAU' else NIVEL
carpeta = DATA / 'zonificación' / f'zonificacion_{NIVEL}'
nombres = leer_primeras_filas(carpeta / f'nombres_{sufijo}.csv')
poblacion = leer_primeras_filas(carpeta / f'poblacion_{sufijo}.csv', cabecera=False)
print('Nombres de zonas:')
display(nombres)
print('Población: column0 = código; column1 = población publicada.')
display(poblacion)

Nombres de zonas:


,column0,ID,name
0,0,01001,Alegría-Dulantzi
1,1,01002,Amurrio
2,2,01004_AM,Artziniega agregacion de municipios
3,3,01009_AM,Asparrena agregacion de municipios
4,4,01010,Ayala/Aiara


Población: column0 = código; column1 = población publicada.


,column0,column1
0,01001,2925.0
1,01002,10307.0
2,01004_AM,3005.0
3,01009_AM,4599.0
4,01010,2951.0


## 7. Correspondencias INE–MITMA y calidad

Leemos el cruce territorial y los distritos descartados para la fecha elegida. El archivo de calidad es distrital, independientemente del nivel seleccionado arriba.

In [8]:
correspondencias = leer_primeras_filas(DATA / 'zonificación' / 'relacion_ine_zonificacionMitma.csv')
calidad = leer_primeras_filas(DATA / 'calidad' / 'diarios' / f'{FECHA}_distritos_descartados.csv')
print('Correspondencias:')
display(correspondencias)
print('Distritos descartados:')
display(calidad)

Correspondencias:
Distritos descartados:


,seccion_ine,distrito_ine,municipio_ine,distrito_mitma,municipio_mitma,gau_mitma
0,0100101001,0100101,01001,01001,01001,01001
1,0100101002,0100101,01001,01001,01001,01001
2,0100201001,0100201,01002,01002,01002,01002
3,0100201002,0100201,01002,01002,01002,01002
4,0100201003,0100201,01002,01002,01002,01002


,fecha,zona
0,20230101,10032_AM
1,20230101,19086_AM
2,20230101,19103_AM
3,20230101,19156_AM
4,20230101,25158_AM


## Nota sobre los archivos geográficos

El inventario incluye polígonos y centroides en formato shapefile. Cada conjunto usa varios archivos (`.shp`, `.shx`, `.dbf`, `.prj`, etc.) que deben permanecer juntos. Su lectura espacial queda fuera de este notebook básico de CSV; no se han transformado ni eliminado.

Las tablas mostradas son solo las primeras filas, no los datasets completos. Para ver más registros, cambia `FILAS` y vuelve a ejecutar desde la celda de selección.